In [ ]:
import torch
import transformers
from datasets import load_dataset
from huggingface_hub import notebook_login

notebook_login()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


name = "answerdotai/ModernBERT-base"
model = transformers.AutoModelForSequenceClassification.from_pretrained(
    name,
    dtype="auto",
    num_labels=2,
).to(device)

tokenizer = transformers.AutoTokenizer.from_pretrained(name)
ds_raw = load_dataset("stanfordnlp/sst2")
ds = ds_raw.map(
    lambda item: tokenizer(
        item["sentence"],
        truncation=True,
        max_length=512,
    )
)

collator = transformers.DataCollatorWithPadding(tokenizer)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [ ]:
for param in model.base_model.parameters():
    param.requires_grad_(False)

training_args = transformers.TrainingArguments(
    output_dir="ModernBert",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = transformers.Trainer(
    model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    processing_class=tokenizer,
    data_collator=collator,
)

ValueError: Your setup doesn't support bf16/gpu. You need to assign use_cpu if you want to train the model on CPU.

In [52]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 